# HFSS 2.45 GHz — Reproducible Field and Shearlet Analysis

This Colab notebook runs the completed, production-validated Gate 1 complex electric-field reconstruction and Gate 2 three-level discrete Shearlet analysis. The complete reproducible flow is: HFSS Real/Imag `.fld` → complex $E_x,E_y,E_z$ → Electric-field magnitude $|E|$ [V/m] → 40 numeric single-channel matrices → unmasked numeric HFSS matrix → reflect padding to 512×512 → three-level Discrete Shearlet Transform → RMS-normalized directional coefficients → Level 1 / Level 2 / Level 3 aggregate responses → `full_phantom` versus `interior_1p5mm` sensitivity analysis → metrics and publication figures.

In [ ]:
# Colab environment
from pathlib import Path
import csv, hashlib, json, subprocess, sys, zipfile, shutil
import numpy as np

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

REPO_URL = "https://github.com/codebycristian-dev/hfss-shearlet-colab.git"
REPO_REF = "main"
REPO_ROOT = Path("/content/hfss-shearlet-colab") if IN_COLAB else Path("..").resolve()
if IN_COLAB and not (REPO_ROOT / "src").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)
if not (REPO_ROOT / "src").is_dir():
    raise RuntimeError(f"Repository src/ directory not found at {REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
git_result = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
GIT_COMMIT_SHA = git_result.stdout.strip() if git_result.returncode == 0 else None
print("Requested repository ref:", REPO_REF)
print("Git commit:", GIT_COMMIT_SHA or "unavailable")


## 1. Data source

The repository code is public and requires no GitHub credentials. The HFSS dataset is **not public** and is not included in the repository. Supply the original ZIP from Google Drive or by manual upload.

Set `DATA_SOURCE = "DRIVE"` or `"UPLOAD"` below.

In [ ]:
DATA_SOURCE = "DRIVE"  # "DRIVE" or "UPLOAD"
DATA_ZIP_NAME = "Entregas_papper(1).zip"

if IN_COLAB and DATA_SOURCE == "DRIVE":
    from google.colab import drive
    drive.mount("/content/drive")
    # CHANGE ONLY THIS PATH if needed:
    DATA_ZIP = Path("/content/drive/MyDrive/HFSS_Dataset") / DATA_ZIP_NAME
elif IN_COLAB and DATA_SOURCE == "UPLOAD":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one dataset ZIP.")
    DATA_ZIP = Path("/content") / next(iter(uploaded))
else:
    DATA_ZIP = Path("../data") / DATA_ZIP_NAME

print("Dataset ZIP:", DATA_ZIP)
if not DATA_ZIP.exists():
    raise FileNotFoundError(DATA_ZIP)

dataset_hasher = hashlib.sha256()
with DATA_ZIP.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        dataset_hasher.update(chunk)
DATASET_SHA256 = dataset_hasher.hexdigest()
print("Dataset filename:", DATA_ZIP.name)
print("Dataset SHA-256:", DATASET_SHA256)


In [ ]:
# Re-extract on every Run All so a previous session cannot supply stale data.
WORK = Path("/content/hfss_work") if IN_COLAB else Path("../.work")
DATA_DIR = WORK / "dataset"
OUTPUT_DIR = WORK / "outputs"

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as archive:
    root = DATA_DIR.resolve()
    for member in archive.infolist():
        target = (DATA_DIR / member.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError(f"Unsafe ZIP member path: {member.filename}")
    archive.extractall(DATA_DIR)

print("Extracted dataset to:", DATA_DIR)


## 2. Gate 1 processing

The pipeline reconstructs

\[
\mathbf E = \Re\{\mathbf E\} + j\Im\{\mathbf E\}
\]

and computes

\[
|\mathbf E|=\sqrt{|E_x|^2+|E_y|^2+|E_z|^2}.
\]

The physical quantity is **Electric-field magnitude $|E|$ [V/m]**. The geometric mask is applied only after the full numeric field is reconstructed. Publication labels are Mode 1 — parallel to y and Mode 2 — perpendicular to y; machine metadata remains `parallel_y` and `perpendicular_y`. All plots preserve equal physical spacing. `article_shared` is for publication-quality visualization only and uses a shared `cividis` scale with a neutral masked exterior. PNG files are visualization artifacts only; PNG/RGB pixels are NEVER transform input. Scientific analysis uses numeric NPZ arrays. No per-image normalization is used.

In [ ]:
from src.pipeline import run_intensity_pipeline

rows = run_intensity_pipeline(
    DATA_DIR, OUTPUT_DIR,
    dataset_filename=DATA_ZIP.name,
    dataset_sha256=DATASET_SHA256,
    git_commit_sha=GIT_COMMIT_SHA,
    repository_ref=REPO_REF,
)
print(f"Processed {len(rows)} cuts.")
assert len(rows) == 40, f"Expected 40 cuts, got {len(rows)}"


In [ ]:
# Gate 1 output validation
physical_pngs = sorted((OUTPUT_DIR / "01_intensity" / "physical_shared").rglob("*.png"))
presentation_pngs = sorted((OUTPUT_DIR / "01_intensity" / "presentation_shared").rglob("*.png"))
article_pngs = sorted((OUTPUT_DIR / "01_intensity" / "article_shared").rglob("*.png"))
article_figures = sorted((OUTPUT_DIR / "03_article_figures").glob("*.png")) + sorted((OUTPUT_DIR / "03_article_figures").glob("*.pdf"))
matrices = sorted((OUTPUT_DIR / "02_numeric").rglob("*.npz"))
metrics = OUTPUT_DIR / "04_metrics" / "field_metrics.csv"
run_metadata_path = OUTPUT_DIR / "04_metrics" / "run_metadata.json"
with metrics.open(newline="", encoding="utf-8") as stream:
    metric_rows = list(csv.DictReader(stream))
run_metadata = json.loads(run_metadata_path.read_text(encoding="utf-8"))
expected_groups = {(f"Mode{mode}", plane) for mode in (1, 2) for plane in ("XZ", "YZ")}
physical_groups = {(path.parent.parent.name, path.parent.name) for path in physical_pngs}
presentation_groups = {(path.parent.parent.name, path.parent.name) for path in presentation_pngs}
matrix_groups = {(path.parent.parent.name, path.parent.name) for path in matrices}

print("Physical shared-scale PNGs:", len(physical_pngs))
print("Presentation shared-scale PNGs:", len(presentation_pngs))
print("Article shared-scale PNGs:", len(article_pngs))
print("Article figures:", len(article_figures))
print("Numeric matrices:", len(matrices))
print("Metric rows:", len(metric_rows))
print("Metrics:", metrics)
assert len(physical_pngs) == 40, f"Expected 40 physical PNGs, got {len(physical_pngs)}"
assert len(presentation_pngs) == 40, f"Expected 40 presentation PNGs, got {len(presentation_pngs)}"
assert len(article_pngs) == 40, f"Expected 40 article PNGs, got {len(article_pngs)}"
assert len(article_figures) == 6, f"Expected 6 article figures, got {len(article_figures)}"
assert len(matrices) == 40, f"Expected 40 numeric matrices, got {len(matrices)}"
assert len(metric_rows) == 40, f"Expected 40 metric rows, got {len(metric_rows)}"
assert physical_groups == expected_groups, f"Invalid physical PNG structure: {physical_groups}"
assert presentation_groups == expected_groups, f"Invalid presentation PNG structure: {presentation_groups}"
assert matrix_groups == expected_groups, f"Invalid matrix structure: {matrix_groups}"
assert {row["polarization"] for row in metric_rows} == {"parallel_y", "perpendicular_y"}
assert run_metadata["dataset_filename"] == DATA_ZIP.name
assert run_metadata["dataset_sha256"] == DATASET_SHA256
assert run_metadata["git_commit_sha"] == GIT_COMMIT_SHA
assert run_metadata["repository_ref"] == REPO_REF
assert run_metadata["frequency_GHz"] == 2.45
assert run_metadata["quantity"] == "Electric-field magnitude |E|"
assert run_metadata["unit"] == "V/m"
print("GATE 1: PASS")


In [ ]:
# Gate 1 checkpoint: Gate 2 runs next and the combined output is packaged at the end.
print("Gate 1 outputs retained in:", OUTPUT_DIR)


# Gate 2 — Three-level Discrete Shearlet Analysis

Gate 2 analyzes the **numeric, single-channel** HFSS NPZ matrices—not PNG pixels or RGB renderings. PNG files are visualization artifacts only, and PNG/RGB pixels are NEVER transform input. The transform input is `unmasked_intensity_V_per_m`, because applying the hard phantom mask before decomposition would introduce an artificial edge at the phantom boundary. No per-image normalization is used; physical V/m values are preserved.

Each rectangular plane is symmetrically reflect-padded to the smallest power-of-two square (512×512 for the production data), with no resize or interpolation. For this backend construction, Level 1 is the coarser spatial-scale band, Level 2 is the intermediate spatial-scale band, and Level 3 is the finer spatial-scale band; exact physical spatial-frequency cutoffs have not been calibrated. Directional filters are grouped from `shearletSystem["shearletIdxs"]` by `[cone, scale, shearing]`, and the single `[0,0,0]` low-pass is excluded. The production system is required to contain exactly 33 filters: 1 low-pass and 8/8/16 directional filters at Levels 1/2/3. Individual coefficients are divided by the system RMS before directional response is aggregated.

The full phantom is the primary ROI. A second `interior_1p5mm` ROI reports boundary sensitivity without changing the transform: it uses a Euclidean distance transform sampled in the physical axis spacings, not a pixel-count erosion.


In [ ]:
from src.shearlet_analysis import run_shearlet_pipeline

shearlet_rows = run_shearlet_pipeline(OUTPUT_DIR, git_commit_sha=GIT_COMMIT_SHA, requested_repo_ref=REPO_REF)
print(f"Computed {len(shearlet_rows)} cut-level rows.")
assert len(shearlet_rows) == 120, f"Expected 120 level metric rows, got {len(shearlet_rows)}"


## Directionality and entropy

Within each scale and ROI, the directional coefficient energy is $E_k=\sum_{x,y}|C_{jk}^{\mathrm{norm}}(x,y)|^2$ and $p_k=E_k/\sum_q E_q$. The reported normalized entropy is $H=-\sum_k p_k\ln(p_k)/\ln(K)$, where $K$ is the number of directional filters at that scale. Thus 0 indicates concentration in one direction and 1 indicates equal response across directions. **Shearlet coefficient energy is a signal-processing quantity based on squared RMS-normalized Shearlet coefficients. It is not electromagnetic energy and is not Poynting intensity.** Both the `full_phantom` and `interior_1p5mm` ROIs are reported in the same 120 cut/level rows.


In [ ]:
# Gate 2 output validation
shearlet_root = OUTPUT_DIR / "05_shearlet"
numeric_artifacts = sorted((shearlet_root / "numeric").rglob("*.npz"))
level_maps = sorted((shearlet_root / "level_maps").rglob("level_*.png"))
level_metrics_path = shearlet_root / "metrics" / "shearlet_level_metrics.csv"
direction_metrics_path = shearlet_root / "metrics" / "shearlet_direction_metrics.csv"
shearlet_metadata_path = shearlet_root / "metrics" / "shearlet_run_metadata.json"
with level_metrics_path.open(newline="", encoding="utf-8") as stream:
    level_metric_rows = list(csv.DictReader(stream))
with direction_metrics_path.open(newline="", encoding="utf-8") as stream:
    direction_metric_rows = list(csv.DictReader(stream))
shearlet_metadata = json.loads(shearlet_metadata_path.read_text(encoding="utf-8"))
numeric_scale_maps = 0
for artifact in numeric_artifacts:
    with np.load(artifact, allow_pickle=False) as data:
        for key in ("scale_energy_1", "scale_energy_2", "scale_energy_3"):
            assert key in data.files and np.isfinite(data[key]).all(), f"Invalid {key} in {artifact}"
            numeric_scale_maps += 1
assert len(numeric_artifacts) == 40, f"Expected 40 Shearlet NPZ artifacts, got {len(numeric_artifacts)}"
assert numeric_scale_maps == 120, f"Expected 120 numeric scale maps, got {numeric_scale_maps}"
assert len(level_maps) == 120, f"Expected 120 Shearlet level maps, got {len(level_maps)}"
assert len(level_metric_rows) == 120, f"Expected 120 Shearlet level metric rows, got {len(level_metric_rows)}"
assert len(direction_metric_rows) == 120, f"Expected 120 directional metric rows, got {len(direction_metric_rows)}"
assert shearlet_metadata["nScales"] == 3
assert shearlet_metadata["shearLevels"] == [1, 1, 2]
assert shearlet_metadata["full"] == 0
assert shearlet_metadata["requested_repo_ref"] == REPO_REF
assert shearlet_metadata["git_commit_sha"] == GIT_COMMIT_SHA
assert shearlet_metadata["actual_total_filters"] == 33
assert shearlet_metadata["lowpass_filters"] == 1
assert len(shearlet_metadata["shearletIdxs"]) == 33
assert shearlet_metadata["actual_filters_per_scale"] == {"1": 8, "2": 8, "3": 16}
assert shearlet_metadata["scale_interpretation"]["1"] == "coarser spatial-scale band"
assert "dominant_cone_full_phantom" in direction_metric_rows[0]
assert "dominant_cone_interior_1p5mm" in direction_metric_rows[0]
assert shearlet_metadata["input_npz_key"] == "unmasked_intensity_V_per_m"
assert shearlet_metadata["dataset_sha256"] == DATASET_SHA256
assert shearlet_metadata["dataset_filename"] == DATA_ZIP.name
assert shearlet_metadata["number_of_numeric_scale_maps"] == 120
assert shearlet_metadata["number_of_level_maps"] == 120
assert shearlet_metadata["maximum_all_cut_reconstruction_error"] < shearlet_metadata["reconstruction_tolerance"]
assert set(shearlet_metadata["software_versions"]) == {"python", "numpy", "scipy", "matplotlib", "pyShearLab-MIND"}
print("GATE 2: PASS")


## Interpretation note

A bright coefficient response near the phantom boundary is not automatically a material feature; compare the full-phantom and interior-1.5-mm metrics. An optional extension is to compare other physically specified interior distances while leaving the decomposition unchanged.


In [ ]:
# Final downloadable archive: Gate 1 and Gate 2 artifacts.
archive_base = str(WORK / "HFSS_Gate1_Gate2_results_2p45GHz")
zip_path = shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Created:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
